# OpenRCA — secondo confronto per il relatore
**A1_VERIFY vs A1_VERIFY_ST | 8 casi development, 16 run | sola lettura**

Questa demo usa risultati **già congelati e valutati**. Non interroga Gemini, non legge `record.csv` o `query.csv` e non avvia runner sperimentali. Gli artefatti sono in Google Drive, il codice sperimentale esatto è il commit `138fb523efec4a665a2424ab2faedb698f10562e`.

**Domanda:** il contesto strutturale/temporale osservativo aggiunto alla *finalizzazione* migliora la diagnosi rispetto ad A1_VERIFY, mantenendo 8 regole e budget 4+2?

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1 — Verifica di integrità del freeze e dello scoring
Confronta i byte dei due manifest con i relativi SHA-256. Un file diverso dal congelato interrompe la demo.

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib, json
import pandas as pd

root = Path("/content/drive/MyDrive/Tesi_OpenRCA")
protocol = "a1_verify_st_paired_development_v1"
freeze_path = root / "data/protocols" / f"{protocol}_predictions_frozen_v1.json"
score_path = root / "data/results" / f"{protocol}_official_scoring_v1.json"
EXPECTED_FREEZE = "43e646792685b1cde03052cadc2c8a82749d058a4e04ee0b01f607a7384fd104"
EXPECTED_SCORING = "26f10169e379f9a8ceabcd83923ae7daa72ca90f705e021d77e6eb5772027f45"
EXPECTED_COMMIT = "138fb523efec4a665a2424ab2faedb698f10562e"

def sha256(raw):
    return hashlib.sha256(raw).hexdigest()

assert sha256(freeze_path.read_bytes()) == EXPECTED_FREEZE
assert sha256(score_path.read_bytes()) == EXPECTED_SCORING
freeze = json.loads(freeze_path.read_text(encoding="utf-8"))
scoring = json.loads(score_path.read_text(encoding="utf-8"))
assert freeze["protocol"] == protocol == scoring["protocol"]
assert freeze["git_commit"] == EXPECTED_COMMIT
assert freeze["run_count"] == 16 and len(freeze["results"]) == 16
assert len(scoring["results"]) == 16
print("Freeze e scoring verificati con SHA-256. Nessuna API chiamata.")

## 2 — Audit read-only delle 16 predizioni
Controllo dei file originali, della provenienza e della parità delle evidenze. La predizione non conforme di `cloudbed-2_q053` resta intatta.

In [ ]:
records = []
pair_sha = defaultdict(list)
seen = set()
for item in freeze["results"]:
    run_id = item["run_id"]
    assert run_id not in seen
    seen.add(run_id)
    path = root / item["result_relative_path"]
    raw = path.read_bytes()
    assert sha256(raw) == item["result_file_sha256"]
    result = json.loads(raw)
    provenance = result["paired_provenance"]
    assert provenance["run_id"] == run_id
    assert provenance["git_commit"] == EXPECTED_COMMIT
    assert provenance["integrity_issues"] == []
    assert result["error"] is None
    assert sha256(result["raw_response"].encode("utf-8")) == item["raw_response_sha256"]
    assert result["variant"] == item["variant"]
    assert result["case_id"] == item["case_id"]
    m = result["metrics"]
    assert m["token_usage_complete"] is True
    assert 0 <= m["exploration_tool_rounds"] <= 4
    assert 0 <= m["verification_tool_rounds"] <= 2
    pair_sha[item["case_id"]].append(result["evidence_sha256"])
    if result["variant"] == "A1_VERIFY_ST":
        assert isinstance(result.get("structural_temporal_context"), dict)
    else:
        assert result.get("structural_temporal_context") is None
    records.append({
        "case_id": item["case_id"],
        "variant": item["variant"],
        "json_status": result["json_status"],
        "tokens": m["total_tokens"],
        "seconds": m["end_to_end_seconds"],
        "checks": m["verification_tool_rounds"],
    })
assert len(pair_sha) == 8 and all(len(v) == 2 and v[0] == v[1] for v in pair_sha.values())
runs = pd.DataFrame(records)
display(runs.sort_values(["case_id", "variant"]).reset_index(drop=True))
print("Audit completato: 8 coppie, 16 risultati e stessa evidenza iniziale per coppia.")

## 3 — Risultati ufficiali già registrati, senza ricalcolare lo scoring
Le diagnosi sono state congelate prima dell'apertura delle etichette. Questa cella legge solo il file di scoring preesistente e separato.

In [ ]:
scores = pd.DataFrame(scoring["results"])
assert set(scores["variant"]) == {"A1_VERIFY", "A1_VERIFY_ST"}
assert not scores.duplicated(["case_id","variant"]).any()
pivot = scores.pivot(index="case_id", columns="variant", values="score")
assert pivot.shape == (8, 2)
pivot["delta_ST"] = pivot["A1_VERIFY_ST"] - pivot["A1_VERIFY"]
display(pivot.round(2))
wins = int((pivot["delta_ST"] > 1e-9).sum())
losses = int((pivot["delta_ST"] < -1e-9).sum())
ties = int((pivot["delta_ST"].abs() <= 1e-9).sum())
print(f"ST: {wins} caso migliorato, {losses} peggiorato, {ties} pari")
print("Score medio VERIFY:", round(pivot["A1_VERIFY"].mean(), 4))
print("Score medio ST:", round(pivot["A1_VERIFY_ST"].mean(), 4))
print("Differenza media ST−VERIFY:", round(pivot["delta_ST"].mean(), 4))
print("Output legacy validi:", runs.assign(valid=runs["json_status"]=="valid_json").groupby("variant")["valid"].sum().to_dict())
print("Token totali:", runs.groupby("variant")["tokens"].sum().to_dict())
print("Secondi totali:", runs.groupby("variant")["seconds"].sum().round(3).to_dict())

## 4 — Grafico delle differenze appaiate
Il delta medio positivo **non** è distribuito uniformemente: ST migliora solo un caso e peggiora un altro.

In [ ]:
import matplotlib.pyplot as plt
ax = pivot["delta_ST"].plot(kind="bar", figsize=(11,4))
ax.axhline(0, linewidth=1)
ax.set_title("Differenza score ufficiale per caso: A1_VERIFY_ST − A1_VERIFY")
ax.set_xlabel("Incidente")
ax.set_ylabel("Delta score")
plt.tight_layout()
plt.show()

## 5 — Limiti scientifici e prossime fasi

- L'implementazione ST attuale confronta segnali **nodo/container** e alcuni ordini temporali, ma **non** ricostruisce né verifica un percorso causale RPC fra microservizi. Non è PAVE completo.
- I due studi preesistenti `A1-R8 vs A1_VERIFY` e `A1_VERIFY vs A1_VERIFY_ST` usano **coorti differenti**: non confrontare i loro score come se fossero lo stesso esperimento.
- Questo è un confronto development di fattibilità (`n=8`), non una valutazione su holdout blind. La conformità JSON non implica correttezza della root cause.
- Le tre regole mancanti R03/R04/R05 per anomalie di rete richiedono prima un audit GT-free dei segnali disponibili in metriche, trace e log.
- Nessuna esecuzione aggiuntiva, nessuna modifica ai risultati; per nuovi ablation test usare un nuovo branch, commit, protocollo e coorte.

**Riferimento repository:** `docs/a1_verify_st_paired_results_2026-11-02.md`.